In [ ]:

def Pt2planeSignedDistance(plane,point): #in utils
    '''
    returns the orthogonal distance of a given point X0 to the plane p in a metric space (projection of X0 on p = P), 
    with the sign determined by whether or not X0 is in the interior of p with respect to the center of gravity [0 0 0]
    - input:
        - plane = [u v w h] definition of the P plane (numpy array)
        - point = [x0 y0 z0] coordinates of the X0 point (numpy array)
    returns the signed modulus ±||PX0||
    '''
    sd = (plane[3] + np.dot(plane[0:3],point))/np.sqrt(plane[0]**2+plane[1]**2+plane[2]**2)
    return sd



def truncateAboveEachPlane(planes: np.ndarray, #in utils
                           coords,
                           debug: bool=False,
                           delAbove: bool=True,
                           noOutput: bool=False):
    '''
    - input: 
        - planes = numpy array with all [u v w d] plane definitions
        - coords = (N,3) numpy array will all coordinates
        - delAbove = if True (default) delete atoms that lie above the planes + eps = 1e-4. Delete atoms below the
                     planes otherwise (use with precaution, could return no atoms as a function of their definition)
        - noOutput = do not print any message
    - returns the indexes of the atoms that are above each input planes
    '''

    delAtoms = []

    eps =1e-3
    for p in planes:
        print('p',p)
        for i,c in enumerate(coords):
            signedDistance = Pt2planeSignedDistance(p,c)
            if delAbove and signedDistance > eps:
                delAtoms.append(i)
            elif not delAbove and signedDistance < eps:
                delAtoms.append(i)
        # print(keptAtoms)
        if debug and not noOutput:
            for a in delAtoms:
                print(f"@{a+1}",end=',')
            print("",end='\n')
    delAtoms = np.array(delAtoms)
    delAtoms = np.unique(delAtoms)
    # if (debug): plot3D()
    return delAtoms




In [ ]:
class hollow_shapes:
    #cube
    nFaces = 6
    nEdges = 12
    nVertices = 8
    edgeLengthFfcc = np.sqrt(2)
    edgeLengthFbcc = 2/np.sqrt(3)
    radiusCSF = np.sqrt(3)/2
    radiusISF = 1/2
    
    def __init__(self,
                 crystalStructure='fcc',
                 element='Au',
                 Rnn: float=2.7,
                 nOrder: int=1,
                 inner_cube_size: int=5, #Angs?
                 outer_cube_size: int=10, #Angs?
                 delAbove: bool= False, # Delete atoms below the planes
                 postAnalyzis=True,
                 aseView: bool=False,
                 thresholdCoreSurface = 1.,
                 skipSymmetryAnalyzis = False,
                 jmolCrystalShape: bool=True,
                 noOutput = False,
                 calcPropOnly = False,
                ):
        self.crystalStructure = crystalStructure
        self.element = element
        self.Rnn = Rnn
        #self.nOrder = nOrder
        self.inner_cube_size = inner_cube_size 
        self.outer_cube_size = outer_cube_size
        self.nAtomsPerEdge = nOrder+1
        self.nAtoms = 0
        self.nAtomsPerShell = [0]
        self.jmolCrystalShape = jmolCrystalShape
        self.cog = np.array([0., 0., 0.])
        #self.imageFile = pyNMBu.imageNameWithPathway("cube-C.png")
        self.trPlanes = None
        # if not noOutput: vID.centerTitle(f"{nOrder}x{nOrder}x{nOrder} {self.crystalStructure} cube")
          
        #if not noOutput: self.prop()
        if not calcPropOnly:
            self.coords(noOutput)
            if aseView: view(self.NP)
            if postAnalyzis:
                self.propPostMake(skipSymmetryAnalyzis,thresholdCoreSurface, noOutput=noOutput)
                if aseView: view(self.NPcs)
          
    def __str__(self):
        return(f" size {self.outer_cube_size} fcc hollow cube with Rnn = {self.Rnn} and inside empty cube of {self.inner_cube_size}  ")
    
    def nAtomsfccF(self,i):
        """ returns the number of atoms of an fcc cube of size i x i x i"""
        return 4*i**3 + 6*i*2 + 3*i + 1

    def nAtomsbccF(self,i):
        """ returns the number of ato # now add last layers
        if not noOutput: print(f"... and adding the upper layers")
        sc = cut(sc,extend=1.05)
        natoms = len(sc.positions)
        self.nAtoms=natomsms of a bcc cube of size i x i x i"""
        return 2*i**3 + 3*i*2 + 3*i
    
    def nAtomsPerShellAnalytic(self):
        n = []
        Sum = 0
        for i in range(self.nOrder+1):
            Sum = sum(n)
            ni = self.nAtomsF(i)
            n.append(ni-Sum)
        return n
    
    def nAtomsPerShellCumulativeAnalytic(self):
        n = []
        Sum = 0
        for i in range(self.nOrder+1):
            Sum = sum(n)
            ni = self.nAtomsF(i)
            n.append(ni)
        return n
    
    def nAtomsfccAnalytic(self):
        n = self.nAtomsfccF(self.nOrder)
        return n
        
    def nAtomsbccAnalytic(self):
        n = self.nAtomsbccF(self.nOrder)
        return n
        
    def edgeLength(self):
        if self.crystalStructure == 'fcc':
            return self.Rnn*self.edgeLengthFfcc*self.nOrder
        elif self.crystalStructure == 'bcc':
            return self.Rnn*self.edgeLengthFbcc*self.nOrder
        
    def latticeConstant(self):
        if self.crystalStructure == 'fcc':
            return self.Rnn*self.edgeLengthFfcc
        elif self.crystalStructure == 'bcc':
            return self.Rnn*self.edgeLengthFbcc
        
    def radiusCircumscribedSphere(self):
        return self.radiusCSF*self.edgeLength()

    def radiusInscribedSphere(self):
        return self.radiusISF*self.edgeLength()

    def area(self):
        el = self.edgeLength()
        return 6 * el**2

    def volume(self):
        el = self.edgeLength()
        return el**3

    #if self.shape=='cube' : 
    def coords(self,noOutput):
        if not noOutput: vID.centertxt("Generation of coordinates",bgc='#007a7a',size='14',weight='bold')
        chrono = pyNMBu.timer(); chrono.chrono_start()
        if self.crystalStructure == 'fcc':
            cube = bulk(self.element, 'fcc', a=self.latticeConstant(), cubic=True)
        elif self.crystalStructure == 'bcc':
            cube = bulk(self.element, 'bcc', a=self.latticeConstant(), cubic=True)
       
        self.n_cells = int(self.outer_cube_size / self.latticeConstant())
        M = [[self.n_cells, 0, 0], [0, self.n_cells, 0], [0, 0, self.n_cells]]
        sc=make_supercell(cube, M)
        if not noOutput: print(f"Now making a {self.n_cells}x{self.n_cells}x{self.n_cells} fcc supercell...")
        # now add last layers
        if not noOutput: print(f"... and adding the upper layers")
        sc = cut(sc,extend=1.05)
        coordsNP = sc.get_positions()
        oldcog = sc.get_center_of_mass()
        coordsNP = coordsNP - oldcog
        sc.set_positions(coordsNP)
        self.NP = sc
        half_inner_cube_size=self.inner_cube_size/2
        #empty the inside cube
        planes_above=[0,0,-1,half_inner_cube_size],[-1,0,0,half_inner_cube_size],[0,-1,0,half_inner_cube_size]]
        planes_under=[[0,0,1,half_inner_cube_size],[1,0,0,half_inner_cube_size],[0,1,0,half_inner_cube_size]
        AtomsUnderPlanes=pyNMBu.truncateAboveEachPlane(planes_above,coordsNP,debug=False, delAbove=True,noOutput=False)
        AtomsUnderPlanes=pyNMBu.truncateAboveEachPlane(planes_under,coordsNP,debug=False, delAbove=False,noOutput=False)
        #self.NP = self.sc.copy()
        del self.NP[AtomsUnderPlanes]


        #nAtoms = self.NP.get_global_number_of_atoms()
        natoms = len(sc.positions)
        self.nAtoms=natoms
        self.cog = pyNMBu.centerOfGravity(sc.get_positions())
        if not noOutput: chrono.chrono_stop(hdelay=False); chrono.chrono_show()
        
       
        self.cog = self.NP.get_center_of_mass()
        #if self.trPlanes is not None: self.trPlanes = pyNMBu.setdAsNegative(self.trPlanes) ?????
        if self.jmolCrystalShape: self.jMolCS = pyNMBu.defCrystalShapeForJMol(self,noOutput) 
            

        
    def prop(self):
        vID.centertxt("Properties",bgc='#007a7a',size='14',weight='bold')
        print(self)
        pyNMBu.plotImageInPropFunction(self.imageFile)
        print("element = ",self.element)
        print("number of vertices = ",self.nVertices)
        print("number of edges = ",self.nEdges)
        print("number of faces = ",self.nFaces)
        print(f"nearest neighbour distance = {self.Rnn:.2f} Å")
        print(f"lattice constant = {self.latticeConstant():.2f} Å")
        print(f"edge length = {self.edgeLength()*0.1:.2f} nm")
        print(f"radius after volume = {pyNMBu.RadiusSphereAfterV(self.volume()*1e-3):.2f} nm")
        print(f"radius of the circumscribed sphere = {self.radiusCircumscribedSphere()*0.1:.2f} nm")
        print(f"radius of the inscribed sphere = {self.radiusInscribedSphere()*0.1:.2f} nm")
        print(f"area = {self.area()*1e-2:.1f} nm2")
        print(f"volume = {self.volume()*1e-3:.1f} nm3")
        # print("number of atoms per shell = ",self.nAtomsPerShellAnalytic())
        # print("cumulative number of atoms per shell = ",self.nAtomsPerShellCumulativeAnalytic())
        if self.crystalStructure == 'fcc':
            print("total number of atoms = ",self.nAtomsfccAnalytic())
        elif self.crystalStructure == 'bcc':
            print("total number of atoms = ",self.nAtomsbccAnalytic())
        print("Dual polyhedron: octahedron")

    def propPostMake(self,skipSymmetryAnalyzis,thresholdCoreSurface, noOutput):
        pyNMBu.moi(self.NP, noOutput)
        if not skipSymmetryAnalyzis: pyNMBu.MolSym(self.NP, noOutput=noOutput)
        [self.vertices,self.simplices,self.neighbors,self.equations],surfaceAtoms =\
            pyNMBu.coreSurface(self,thresholdCoreSurface, noOutput=noOutput)
        self.NPcs = self.NP.copy()
        self.NPcs.numbers[np.invert(surfaceAtoms)] = 102 #Nobelium, because it 